# Bai tap:

# Xac dinh tin hieu Buy hoac Ban theo chien luoc cua ban
# P/s: Su dung talib

### Ví dụ: đánh fx khung 1h
### Giá tăng nằm trên (MA6 cắt lên MA10 cắt trên MA50)
### Đường MACD cắt lên
### Vẽ đường kênh giá qua các đỉnh (nếu giá phá qua đường kênh giá) => Phai biet cach lay dinh nhu the nao
###  —> tín hiệu mua
######  Giá cắt xuống (MA6 dưới MA10, MA10 dưới MA50) MACD cắt xuống

—> tín hiệu bán

In [ ]:
# Lay khung 1h
# Tin hieu mua: MA6 > MA10 va MA10 > MA50 va Close > MA6 va MACD > Signal Line
# Tin hieu ban: MA6 < MA 10 va MA10 < MA50 va Close < MA6 vaf MACD < Signal Line

# Load data

In [3]:
def loaddataMT5_FromTo(symbol, from_date, to_date, timeframe):
    import MetaTrader5 as mt5
    from datetime import datetime
    import pandas as pd 

    # Kết nối tới MetaTrader 5
    if not mt5.initialize():
        print("Khởi tạo MT5 không thành công")
        mt5.shutdown()

    from_date_str = datetime.strptime(from_date, '%Y-%m-%d')
    to_date_str = datetime.strptime(to_date, '%Y-%m-%d')
    
    # Lấy dữ liệu OHLC cho cặp tiền symbol trong khoảng thời gian đã xác định
    ohlc_data = mt5.copy_rates_range(symbol, timeframe, from_date_str, to_date_str)
    # print('OHLC_Data')
    # print(ohlc_data)
    # Ngắt kết nối với MT5
    mt5.shutdown()

    # Chuyển dữ liệu nhận được thành DataFrame và hiển thị
    data = pd.DataFrame(ohlc_data)
    data['time'] = pd.to_datetime(data['time'], unit='s')

    # ohlc_df.reset_index(inplace=True)

    data = data.rename(columns={'time': 'Datetime'})        
    data = data.rename(columns={'open': 'Open'})       
    data = data.rename(columns={'high': 'High'})       
    data = data.rename(columns={'low': 'Low'})       
    data = data.rename(columns={'close': 'Close'})       
    data = data.rename(columns={'tick_volume': 'Volume'})       

    data = pd.DataFrame(data, columns=['Datetime', 'Open', 'High', 'Low', 'Close', 'Volume'])

    return data

In [2]:
# import pandas as pd
# import plotly.graph_objects as go
# import redis
# import numpy as np
# from datetime import datetime, timedelta
# import MetaTrader5 as mt5

# # ##############################################Step 0: Định nghĩa các tham số##############################################
# symbol = 'EURUSDs'
# from_date = (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%d')  # Lấy ngày hôm qua
# to_date = (datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d')
# timeframe = mt5.TIMEFRAME_H1

# # ##############################################Step 1: Lấy dữ liệu##############################################
# data = loaddataMT5_FromTo(symbol, from_date, to_date, timeframe)
# data

# Tinh toan chien luoc

In [5]:
import pandas as pd
import plotly.graph_objects as go
import redis
import numpy as np
from datetime import datetime, timedelta
import MetaTrader5 as mt5 

# ##############################################Step 0: Các tham số để lấy dữ liệu###############################
symbol = 'EURUSDs'
from_date = (datetime.now() - timedelta(days=5)).strftime('%Y-%m-%d')  # Lấy ngày hôm qua
to_date = (datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d')
timeframe = mt5.TIMEFRAME_H1

# ##############################################Step 1: Lấy dữ liệu##############################################
data = loaddataMT5_FromTo(symbol, from_date, to_date, timeframe)

# ##############################################Step 2: Chiến lược##############################################  
ma6 = 6 
ma10 = 10
ma50 = 50

import talib
data['MA_6'] = talib.SMA(data['Close'], timeperiod=ma6)
data['MA_10'] = talib.SMA(data['Close'], timeperiod=ma10)
data['MA_50'] = talib.SMA(data['Close'], timeperiod=ma50)

data['MACD'], data['Signal_Line'], data['Histogram'] = talib.MACD(data['Close'], fastperiod=12, slowperiod=26, signalperiod=9)

# Xác định tín hiệu mua
data['Buy_Signal'] = (data['MA_6'] >= data['MA_10']) & (data['MA_10'] >= data['MA_50']) & (data['Close'] >= data['MA_6']) & (data['MACD'] >= data['Signal_Line'])
# Xác định tín hiệu bán
data['Sell_Signal'] = (data['MA_6'] < data['MA_10']) & (data['MA_10'] <= data['MA_50']) & (data['Close'] < data['MA_6']) & (data['MACD'] < data['Signal_Line'])

data

,Datetime,Open,High,Low,Close,Volume,MA_6,MA_10,MA_50,MACD,Signal_Line,Histogram,Buy_Signal,Sell_Signal
0,2024-10-02 17:00:00,1.10511,1.10563,1.10353,1.10362,5284,NaN,NaN,NaN,NaN,NaN,NaN,False,False
1,2024-10-02 18:00:00,1.10363,1.10503,1.10325,1.10494,3804,NaN,NaN,NaN,NaN,NaN,NaN,False,False
2,2024-10-02 19:00:00,1.10495,1.10548,1.10437,1.10465,2697,NaN,NaN,NaN,NaN,NaN,NaN,False,False
3,2024-10-02 20:00:00,1.10466,1.10500,1.10446,1.10470,2202,NaN,NaN,NaN,NaN,NaN,NaN,False,False
4,2024-10-02 21:00:00,1.10470,1.10485,1.10377,1.10383,1948,NaN,NaN,NaN,NaN,NaN,NaN,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,2024-10-08 13:00:00,1.09873,1.09920,1.09822,1.09853,2451,1.098812,1.098628,1.097811,0.000264,0.000132,0.000132,False,False
93,2024-10-08 14:00:00,1.09852,1.09896,1.09785,1.09876,2419,1.098885,1.098666,1.097732,0.000264,0.000158,0.000106,False,False
94,2024-10-08 15:00:00,1.09876,1.09897,1.09715,1.09759,3209,1.098757,1.098608,1.097628,0.000168,0.000160,0.000008,False,False
95,2024-10-08 16:00:00,1.09757,1.09840,1.09745,1.09779,4278,1.098467,1.098555,1.097519,0.000107,0.000150,-0.000042,False,False


In [6]:
data.to_csv('Buoi 5.Bai tap tai lop.csv')